# Walleye Capital / Quantic - QR Case Study

Thanks for your interest in Walleye Capital and Quantic! Before reading further, please save a copy of this colab notebook to your personal Google drive (or local machine) and do your work there. You cannot save to this location and we don’t want your work to be lost.

Attached is a [brief (three-page) mock paper](https://github.com/wwang-walleye/walleye-coding-test-data/blob/main/ct-3-case-study.pdf) looking at the link between social network sentiment and future returns. We'd like you to use this paper as a starting point in your own analysis and hopefully find something more interesting.

Specifically, we'd like you to:
1. Read the paper and reproduce its results. You don't need to match details exactly, but your results should be (at least) directionally similar.
2. Get better results. The paper constructs signals in a very simplistic and flawed way. Think, experiment, and iterate on how you can construct a better model.
3. Measure your results. The paper measures signal performance in a purely statistical fashion looking at regression coefficients and t-stats. Are there other financial metrics you want to look at when measuring performance?
4. Discuss your work, what you did, and what you would like to explore further given more time and resources. This is your place to show off your creativity with ideas that you don't actually have to implement!

Logistics:
* This is fully "open-note". Feel free to use any non-living resource you want, including Google and ChatGPT. (Of course, you should understand and be able to explain every line you write.) Please do not use any code that you don't own the rights to re-distribute.
* You can install and import additional libraries.
* Please submit a Jupyter notebook in either Python or R as your final output. Your notebook should show your work at every stage and be interspersed with your notes / discussion as appropriate.
* You can submit it by downloading a copy of your notebook and e-mailing it back to us.
* The paper is a mock paper and the dataset is synthetic. If you've looked at this kind of data before, things might not line up exactly with your past results.

From a "test perspective", please consider:
* We are more interested in your work and process than the specific model you arrive at or its exact performance.
* We like high-quality code. High-quality code is legible, standard-conformant, performant, testable, extensible, well-documented, and correct.
* We like creativity - a lot.
* Please don't waste time manually implementing things to show off that you know how to do so (e.g. "manually" doing linear regression). At the same time, we won't be impressed by just throwing modern packages / libraries at the problem unless its clear you are using them for the right reasons.
* This isn't meant to be tricky or annoying. It's meant to simulate a real alpha research project. You'll do better approaching it naturally than trying to find the trick or reverse engineering the problems.
* This test is structured such that with reasonable assumptions you should arrive at reasonable answers. If you find yourself with a process or result that isn’t reasonable for mid-frequency equity research (MF), you may have made a mistake.
* If you come from a different background (e.g. HFT or OMM) think about how MF might be different and what kinds of results are reasonable.
* And finally, please do have fun!

Best of luck. We look forward to reviewing your work.

**NOTE: in order to use this notebook below, please choose whether you want to code in Python or R.**

You can choose the runtime environment using Edit -> Notebook Settings

In [ ]:
# Feel free to import anything you might want to use, beyond the below.
# We've included some helpful defaults for Python or R that you may want to use as a starting point.

##############################
# Python
##############################
# import numpy as np
# import pandas as pd
# import seaborn as sns
# import statsmodels.api as sm
# import statsmodels.formula.api as smf
# from tqdm import tqdm



##############################
# R
##############################
# require(data.table)

# 1. Paper Replication
Below, we download three dataframes with tickers + reference data, daily returns, and features.

Dates:
* The ticker and feature dataframes are dated with the "data date". This is the date corresponding to when the observation happened (_t - 1_). We recieve the data the next morning (_t_) and can enter trades during market hours.
* The return dataframe gives _close-to-close_ forward residual returns. So, the return on date _t_ is the return from the close on _t_ to the close on _t + 1_. Residual returns are the component of returns unexplained by common risk factors like beta, size, etc.

Here are the columns and types for each dataframe. All rows are guaranteed to have a valid and unique pair of date / tickerid.

* Tickers

| Columns | Data Type | Description |
| --- | --- | --- |
| date | DATE | Observation reflects data as of market close on this day. |
| ticker | INTEGER | Company identifier |
| gic2 | INTEGER | Industry classification corresponding to the GIC industry system |
| marketcap | REAL | Market cap in millions of USD. |
| averagedailytradingvolume | REAL | Average daily trading volume in millions of USD. |
| earnings_to_price_ratio | REAL | Earnings to Price ratio |
| free_cash_flow_to_enterprise_value | REAL | LTM Free Cash Flow to Enterprise Value ratio |
| ebit_to_sales | REAL | LTM EBIT / Sales |
| ebitda_to_assets | REAL | LTM EBITDA / Total Assets |

* Returns

The return dataframe gives close-to-close forward residual arithmetic returns. So, the return on date t is the return from the close on t to the close on t + 1. Residual returns are the component of returns unexplained by common risk factors like beta, size, etc. Arithmetic returns are defined as (_(Price(t + k) / Price(t)) - 1_) - these are not logarithmic returns.

| Columns | Data Type | Description |
| --- | --- | --- |
| date | DATE | Observation reflects data as of market close on this day. |
| ticker | INTEGER | Company identifier |
| d1 | REAL | 1-day forward close-to-close residual arithmetic return|


* Features

| Columns | Data Type | Description |
| --- | --- | --- |
| date | DATE | Observation reflects data as of market close on this day. |
| ticker | INTEGER | Company identifier |
| gamma_sentiment | REAL | Sum of message-level sentiment from all relevant messages over the past 24-hours. |
| gamma_volume | REAL | Total number of relevant messages over the past 24-hours. |
| gamma_writers | REAL | Unique users who wrote relevant messages over the past 24-hours. |
| theta_sentiment | REAL | The same values, but from the "theta" social network. |
| theta_volume | REAL | The same values, but from the "theta" social network. |
| theta_writers | REAL | The same values, but from the "theta" social network. |

Please use the "gamma" dataset until asked to do so.

In [1]:
# If you choose to work in Python, uncomment and run this cell.


import pandas as pd

def _get_github_data(file, n_parts=4):
    urls = [
        f"https://raw.githubusercontent.com/wwang-walleye/walleye-coding-test-data/main/ct-data-3-{file}_{i}.csv.gz"
        for i in range(1, n_parts + 1)
    ]

    return pd.concat([pd.read_csv(url, compression="gzip") for url in urls]).reset_index(drop=True)

tickers = _get_github_data("tickers")
returns = _get_github_data("returns")
features = _get_github_data("features")

In [ ]:
# If you choose to work in R, uncomment and run this cell.


# require(data.table)

# getGithubData <- function (file, nPaths=4) {
#     urls <- sprintf("https://raw.githubusercontent.com/wwang-walleye/walleye-coding-test-data/main/ct-data-3-%s_%s.csv.gz", file, 1:nPaths)
#     rbindlist(lapply(urls, fread))
# }

# tickers <- getGithubData("tickers")
# returns <- getGithubData("returns")
# features <- getGithubData("features")

# 1. Paper Replication

## 1.1
First, create a single dataframe correctly combining all three dataframes above.

Hints:
* A good choice of organization here will pay dividends across this test.
* Be careful how you align dates here between returns and features. Perhaps read the date definitions above again if you need a refresher. Think about how a misalignment here may affect your results downstream as a sanity check. As a hint, simply joining the three dataframes on “date” as-is is wrong.
* We expect you to follow idiomatic coding patterns for the libraries you choose to use.

## 1.2
Now, re-create the X (right-hand side) and y (left-hand side) variables from the paper. Be sure to perform exploratory analysis and clean-up data issues. As a hint, there are data issues and some values are just plain wrong.

## 1.3
Recreate the models from the paper. If your results are substantially different from those in the paper, explain why.

# 2. Guided Enhancements

Remember the scientific method. If you’re going to claim something is better, you should bring evidence!

## 2.1
Can you come up with a better way to define abnormal volume? Think about what this metric is trying to capture and then experiment.

## 2.2
The paper uses a five day window to calculate sentiment over and defines abnormal volume as the ratio of volume over the past five days to the past 63 days. Can you improve your results by optimizing these parameters?

## 2.3
Replicate the above work to include the "theta" dataset as well.

## 2.4
Create a single "alpha" by combining all your features from above.

An alpha is a single feature which predicts forward returns. Specifically, in this case, you should select and/or combine your individual features in an optimal way.

# 3. Analysis

## 3.1
Graphically explore your final features and alpha. What do you observe / learn?

## 3.2
Calculate the returns from your alpha factor in a reasonable and correct way. That is, the return someone trading on this alpha will (approximately) realize. As a hint, your answer should be a single timeseries.  

## 3.3
If you were a portfolio manager evaluating these signals, what metrics would you like to see? Calculate these metrics and discuss your findings. How do these compare with purely statistical metrics like t-stats? Which do you prefer?

# 4. Free Exploration and Discussion

## 4.1
Now that you understand the data and have the basics done, improve your alpha. Feel free to do anything you want here as long as it reasonably fits the goal of improving alpha. As a reminder, we will be reviewing this portion with a focus on creativity and research process.

## 4.2
What are some limitations of this research? What would you do differently give more resources (time, compute, people), etc.?

If you have strong feelings about this test (good or bad), do share. If you don't think this test was a good demonstrations of your skills / capabilities, feel free to say so and why.